# CEM4644 - MP4: Segmentation for a quantity take-off

## Workshop (in class): *Three 1940 USDA farmhouse plans*

**No coding needed.** Each grey box below is one *step*: click the (play) button at its left, wait until it finishes,
look at the result, then answer the report question that follows. Run the steps **from top to bottom**.

**What you will do (about 90 minutes)**
1. Meet SAM 3 on an ordinary site photo: ask by name, draw a box, tap an object. Then look at the three drawings.
2. Ask for rooms, doors and windows **by name** on a drawing, and check the rooms against what is really on the drawing.
3. Do the **take-off**: set the scale from a printed dimension, then measure every room in square feet - one cell per drawing.
   Then box **one** door and let SAM 3 find all the others.
4. Look at where the model goes wrong: the words, the weak regions, and words of your own.
5. Try a drawing of your own.

**Before you start:** menu *Runtime -> Change runtime type -> T4 GPU -> Save*. The model used here (SAM 3) is large:
with a GPU each request takes well under a second; without one the steps that need the model take about a minute each.

Everything is in **feet and square feet**. There is no scale printed on a drawing that you can trust blindly: you set
the scale yourself, from a dimension the drawing prints or from something whose real size you know.

In [ ]:
#@title ▶ Step 0 · Run me first (2-3 minutes) { display-mode: "form" }
#@markdown Click the play button and wait for the 'Ready' line. This downloads the drawings and what is really on them and loads SAM 3 (about 3 GB).
#@markdown Untick *load_model* only if you have no GPU and want to skip the steps that need the live model.
load_model = True #@param {type:"boolean"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp4_segmentation", "aec_seg"
FOLDERS = ["mp4_segmentation"]               # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)     # also trims a full copy left by an earlier run
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_seg import lab
lab.setup(dataset="workshop", load_model=load_model)


## Part 1 · Meet SAM 3

Detection (MP3) draws a **box** around an object. **Segmentation** goes one step further: it decides, *pixel by pixel*,
what belongs to the object. Count the pixels and you have an area; know the scale and you have square feet. That is what
makes it useful for a **quantity take-off**.

The model is **SAM 3** (Segment Anything Model 3, Meta 2025). You do not train it, and it has no fixed list of classes.
You tell it *what* or *where*, in one of four ways:

- a **phrase**, such as *helmet* or *wet concrete*: it returns every region that matches, each with a **confidence**;
- a **box** around one object: it cuts out that object's exact outline;
- a **click** on one object: the same thing, from a single point;
- a box around one object **as an example**: it returns every object on the picture that looks like it. No words,
  and that is how a count is made (Step 3d).

Two steps on an ordinary site photo first, so that you see what the model does before it meets a drawing.

In [ ]:
#@title ▶ Step 1a · Ask by name { display-mode: "form" }
#@markdown Pick a phrase, or type your own in *own_phrase* (it wins when it is not empty). Three panels: the photo, the mask (white = the model says *this is it*), the overlay. Try a thing (*helmet*), a material (*wet concrete*), a part (*hand*), and something that is not in the photo at all. Watch the confidences.
photo = "pour: Workers pouring and levelling concrete on a slab" #@param ["pour: Workers pouring and levelling concrete on a slab", "mixer: A mixer truck delivering concrete next to a brick house"]
phrase = "person" #@param ["person", "helmet", "safety vest", "boots", "hose", "rebar", "wet concrete", "hand", "truck", "wheel", "brick wall", "sky", "window"]
own_phrase = "" #@param {type:"string"}
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.intro_phrase(photo, phrase, own_phrase, confidence)


In [ ]:
#@title ▶ Step 1b · Box it, or tap it { display-mode: "form" }
#@markdown Draw a box around an object and label it *box*; or draw a tiny box on an object and label it *point* (its centre is the click). Draw several, click *Submit*: SAM 3 cuts out one object per box or click, no words needed. Needs the live model.
photo = "pour: Workers pouring and levelling concrete on a slab" #@param ["pour: Workers pouring and levelling concrete on a slab", "mixer: A mixer truck delivering concrete next to a brick house"]
lab.intro_draw(photo)


### The drawings

Three small farmhouse floor plans published by the U.S. Department of Agriculture in 1940 (public domain). Black walls, drawn windows and door swings, a printed size inside most rooms and overall dimension lines along two sides. There is no scale bar on any of them: you set the scale yourself from a printed dimension. Every room's real area was measured off each drawing, so the notebook can check your measurements.

Read a drawing before you measure it: the room names and the printed room sizes, the overall dimension lines along two
sides (that is where your scale comes from), the black walls, the windows drawn as a gap with thin lines in it, and the
quarter-circle arcs that are door swings.

In [ ]:
#@title ▶ Step 1c · Browse the drawings { display-mode: "form" }
#@markdown *all drawings* shows all three with what is on them and what you will take off from each. These facts are what is really on each drawing, which the notebook checks your measurements against.
drawing = "all drawings" #@param ["all drawings", "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
lab.show_sheets(drawing)


## Part 2 · Ask for something by name

Pick a drawing and a thing. You get three panels: the drawing, the **mask** (white = the model says *this is it*), and
the **overlay**. Below them: how many regions, and how they compare with **what is really on the drawing**. The **confidence slider** hides the regions the model is unsure about: watch the count change.

In [ ]:
#@title ▶ Step 2a · Original, mask, overlay { display-mode: "form" }
#@markdown Try *room (any)* first, then *bedroom*, then *kitchen*, *porch*, *door*, *window*. Some words work, some find nothing at all: that is the lesson of this part.
drawing = "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
thing = "room (any)" #@param ["room (any)", "bedroom", "kitchen", "living room", "bathroom", "porch", "closet", "door", "window"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.segment(drawing, thing, confidence)


In [ ]:
#@title ▶ Step 2b · Hits, misses and extras { display-mode: "form" }
#@markdown The real ones marked on the drawing: green = a real one the model found, red outline = a real one it missed, blue = a region that is not one at all.
drawing = "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
thing = "room (any)" #@param ["room (any)", "bedroom", "kitchen", "living room", "bathroom", "porch", "closet", "door", "window"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.count(drawing, thing, confidence)


> ### 📝 Report question 1
> From Steps 2a and 2b: which words found what they should (rooms? bedrooms? kitchens? doors? windows?), and which found nothing or something else? These drawings have their rooms measured, so the found / missed / extra overlay of Step 2b works on the room words: give the found / missed / extra numbers for two room words on one drawing at confidence 0.3. Then say what the words that found nothing have in common.

## Part 3 · The take-off

A phrase is quick, but a take-off needs control, so now **you** draw the boxes. One cell per drawing. In each cell, pick
the label above the picture before you draw, and draw in this order:

1. **the scale**: one box exactly along the printed overall dimension, from arrowhead to arrowhead. Only the length along
   that dimension is used. *A 1 % error in the scale is a 2 % error in every area, because area is scale squared.*
2. **the second dimension** down the side of the plan, as a check. The two readings never agree exactly, and the report
   tells you by how much they differ and which one is used.
3. **every room**: a tight box each, the edges on the **inside faces** of the walls - rooms, porches and halls.

Then press *Submit*.

The take-off cells measure rooms and nothing else. Counting is a separate job, and it is a job for the model, not for
your pencil: boxing every door yourself and ticking the boxes off against the real doors would tell you nothing about
SAM 3. So Step 3d does it the model's way - you draw **one** box around one door and SAM 3 finds all the others -
and the notebook tells you how many of the real doors it got. On the homework's structural and MEP sheets the same one box counts
footings and light fixtures.

How a *room* box becomes square feet: a box on its own makes SAM 3 cut out the *furniture symbols* inside it rather than
the floor (it was trained to find objects). So the notebook asks for *empty room* **and** hands it your box, keeps the
region that fits your box, fills the holes the symbols leave, removes the black walls, gives back the bites that door
swings take out of a rectangular room, and converts the pixels with **your** scale.

In [ ]:
#@title ▶ Step 3a · Take-off: usda_5544, Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544) { display-mode: "form" }
lab.takeoff("usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)")


In [ ]:
#@title ▶ Step 3b · Take-off: usda_5540, Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540) { display-mode: "form" }
lab.takeoff("usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)")


In [ ]:
#@title ▶ Step 3c · Take-off: usda_5539, Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539) { display-mode: "form" }
lab.takeoff("usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)")


> ### 📝 Report question 2
> From Step 3 on all three drawings: your two scale readings on each drawing, how far each one is from the drawing's known scale and how far they are from each other; and the room table (your square feet, the drawing's, the error) for the drawing you did best on. What is the total of your rooms against the drawing's indoor total?

> ### 📝 Report question 3
> Which rooms came out worst, and why? Look at the pictures and name the reason for at least three of them (a loose box, a kitchen counter or a bathtub eaten out of the mask, a hall that is really a set of doorways, an open space with no wall on one side, the scale). Did the notebook warn you about any of them, and was the warning right?

### Box one, and SAM 3 finds the rest

In Step 2 the word *door* found nothing. Now give the model an **example** instead of a word: one box around one door,
the opening and its swing arc together. SAM 3 looks at what is inside your box and returns everything on the sheet that
looks like it - the fourth way of asking from Part 1. The picture marks what it found and what it missed, so the
count is checked, not taken on trust. Which door you pick matters: a clean single door in a quiet spot is a good
example; a double door or a closet door in a cluttered corner is a poor one, and the count drops.

In [ ]:
#@title ▶ Step 3d · Box one door, and SAM 3 finds the rest { display-mode: "form" }
drawing = "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
lab.find_like(drawing)


> ### 📝 Report question 4
> From Step 3d: on each of the three drawings, the best count you got from one example box (found / missed / extra, and the confidence), and how much the count changed when you picked a different door as the example. Which drawing was hardest and why? In Step 2 the word *door* found nothing: why does one example work where the word does not?

## Part 4 · Where it goes wrong

Three kinds of error to look for: the **words** you use (the model was trained on everyday photographs, not on drawings),
the **weak regions** it proposes with a low confidence, and words of your own that describe what is *drawn* rather than
what it *means*.

In [ ]:
#@title ▶ Step 4a · Does the wording matter? { display-mode: "form" }
#@markdown The same thing asked for with three or four different words. All the wordings are precomputed, so this is instant. Try *door* (against *curved line*) and *window* (against *short parallel lines*).
drawing = "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
thing = "door" #@param ["room (any)", "bedroom", "kitchen", "living room", "bathroom", "porch", "closet", "door", "window"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.phrase_lab(drawing, thing, confidence)


In [ ]:
#@title ▶ Step 4b · Look at each region and its confidence { display-mode: "form" }
#@markdown Every region the model proposed, numbered, with its confidence, its area and the room it sits on. Move the slider to see which ones survive.
drawing = "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
thing = "room (any)" #@param ["room (any)", "bedroom", "kitchen", "living room", "bathroom", "porch", "closet", "door", "window"]
lab.inspect(drawing, thing)


In [ ]:
#@title ▶ Step 4c · Your own words { display-mode: "form" }
#@markdown Type any phrase: a room, a symbol, a shape. Try *curved line* (the door swings), *thick black line* (the walls), *circle*, *small rectangle*, *hatched square*. Needs the live model.
drawing = "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
phrase = "curved line" #@param {type:"string"}
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.your_phrase(drawing, phrase, confidence)


> ### 📝 Report question 5
> From Step 4a: which wording worked best for the thing you chose, and how different were the counts? From Step 4b: describe one weak region (what it sits on, its confidence) and one plain mistake. What would you tell a colleague who wants to put these square feet in a cost estimate?

> ### 📝 Report question 6
> From Step 4c: which of your own words found something that the name of the thing could not (for example *curved line* for the door swings, *thick black line* for the walls)? Why does a shape word work on a drawing where the name of the thing does not?

## Part 5 · Your own drawing

In [ ]:
#@title ▶ Your drawing, your words { display-mode: "form" }
#@markdown This cell prints a **link**: open it in a new tab (it works on a phone too). Upload a drawing, then ask the three ways of this lab: a box for the scale (a printed dimension, plus its length in feet), a box for a room or an object, one example box that SAM 3 finds the rest of, or a phrase. A box is two clicks on the drawing: top-left, then bottom-right. Test at least one drawing of your own and take screenshots. Needs the live model.
lab.upload_app()


> ### 📝 Report question 7
> Test one drawing of your own (any floor plan or construction drawing). Which phrase or box did you use, what did it find, and was the result right? Then: where in a project would a take-off like this be useful, and where would it mislead you? What would you need (clean drawings, a known dimension, a room schedule, a person checking) before you would put these numbers in an estimate?

## Wrap-up

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
#@markdown Every take-off you submitted in Part 3, printed again in one place.
lab.report_summary()


### Where the drawings come from, and the model

- **usda_5544** - USDA design 710-5544, 'Five-room farmhouse', Miscellaneous Publication 360 'Plans of Farm Buildings for Southern States' (1940), p. 17. U.S. Department of Agriculture, Bureau of Agricultural Engineering. Public domain (work of the U.S. Government, 17 U.S.C. 105). https://archive.org/details/plansoffarmbuild360unit
- **usda_5540** - USDA design 710-5540, 'Four-room and attic farmhouse', Miscellaneous Publication 360 'Plans of Farm Buildings for Southern States' (1940), p. 13. U.S. Department of Agriculture, Bureau of Agricultural Engineering. Public domain (work of the U.S. Government, 17 U.S.C. 105). https://archive.org/details/plansoffarmbuild360unit
- **usda_5539** - USDA design 710-5539, 'Four-room farmhouse', Miscellaneous Publication 360 'Plans of Farm Buildings for Southern States' (1940), p. 12. U.S. Department of Agriculture, Bureau of Agricultural Engineering. Public domain (work of the U.S. Government, 17 U.S.C. 105). https://archive.org/details/plansoffarmbuild360unit
- Site photos in Part 1: Workers pouring and levelling concrete on a slab (U.S. Air Force / Airman Sydney Franklin, Public domain, https://upload.wikimedia.org/wikipedia/commons/0/0e/Concrete_pouring_for_the_new_Spangdahlem_Elementary_School_%288062807%29.jpg); A mixer truck delivering concrete next to a brick house (Kolforn, CC BY-SA 4.0, https://upload.wikimedia.org/wikipedia/commons/5/57/-2021-01-18_Foundations_and_concrete_oversite%2C_Trimingham%2C_Norfolk_%283%29.JPG).
- Model: SAM 3 by Meta AI (SAM License), loaded from a public mirror of the official checkpoint; a copy of the licence is in `docs/SAM_LICENSE.txt`.
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp4_segmentation`).